In [1]:
# pytorch 8

# optuna : hyperparams tuning

# complete fMNIST dataset (70000 images)

In [3]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset,DataLoader
import torchmetrics

import optuna

In [4]:
# !pip install torchmetrics

# !pip install optuna

In [5]:
df = pd.read_csv('/content/drive/MyDrive/fmnist_full_dataset/fashion-mnist_train.csv')

df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [6]:
X = df.drop('label',axis=1).values
y = df['label'].values

In [7]:
# scaling

X = X/255.0

In [8]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [9]:
# CustomDataset class

class CustomDataset(Dataset):
  def __init__(self,x,y):
    self.x = torch.tensor(x,dtype=torch.float32)
    self.y = torch.tensor(y,dtype=torch.long)

  def __len__(self):
    return len(self.x)

  def __getitem__(self, idx):
    return self.x[idx],self.y[idx]

In [10]:
dataset = CustomDataset(X,y)

In [14]:
# network dynamic arch

class Network(nn.Module):

  def __init__(self,
               input_dim,
               output_dim,
               n_hidden_layers,
               n_neurons,
               dropout_rate):

    super().__init__()

    layers = []

    for i in range(n_hidden_layers):
      layers.append(nn.Linear(input_dim,n_neurons))
      layers.append(nn.BatchNorm1d(n_neurons))
      layers.append(nn.ReLU())
      layers.append(nn.Dropout(dropout_rate))
      input_dim = n_neurons

    layers.append(nn.Linear(input_dim,output_dim))


    self.model = nn.Sequential(*layers)

  def forward(self,x):
    return self.model(x)



In [15]:
# optuna objective

from sklearn.model_selection import train_test_split
import torchmetrics

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.CrossEntropyLoss()

def objective(trial):

    n_hidden_layers = trial.suggest_int('n_hidden_layers', 1, 5)
    n_neurons = trial.suggest_int('n_neurons', 8, 128, step=8)
    n_epochs = trial.suggest_int('n_epochs', 10, 50, step=10)
    optimizer_name = trial.suggest_categorical('optimizer', ['adam','sgd','rmsprop'])
    learning_rate = trial.suggest_float('learning_rate', 1e-5, 1e-2, log=True)
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5, step=0.1)
    batch_size = trial.suggest_categorical('batch_size', [32,64,128])
    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True)

    # ---- Train/Val Split ----
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    train_ds = CustomDataset(X_train, y_train)
    val_ds   = CustomDataset(X_val, y_val)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size)

    # ---- Model ----
    model = Network(
        input_dim=784,
        output_dim=10,
        n_hidden_layers=n_hidden_layers,
        n_neurons=n_neurons,
        dropout_rate=dropout_rate
    ).to(device)

    # ---- Optimizer ----
    if optimizer_name == 'adam':
        optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    elif optimizer_name == 'sgd':
        optimizer = optim.SGD(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    else:
        optimizer = optim.RMSprop(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)

    # ---- Training ----
    for epoch in range(n_epochs):
        model.train()
        for batch_X, batch_y in train_loader:

            batch_X = batch_X.view(-1, 784).to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()
            logits = model(batch_X)
            loss = criterion(logits, batch_y)
            loss.backward()
            optimizer.step()

    # ---- Validation ----
    model.eval()
    accuracy.reset()

    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X = batch_X.view(-1, 784).to(device)
            batch_y = batch_y.to(device)

            logits = model(batch_X)
            preds = torch.argmax(logits, dim=1)
            accuracy.update(preds, batch_y)

    return accuracy.compute().item()

In [16]:
study = optuna.create_study(direction='maximize')

study.optimize(objective,n_trials=10)

print(f"Best params : {study.best_params}")
print(f"Best Accuracy : {study.best_value}")

[I 2026-02-01 18:46:34,573] A new study created in memory with name: no-name-d18b80ac-05ca-4be3-9c34-54b11988119a
[I 2026-02-01 18:46:51,051] Trial 0 finished with value: 0.796500027179718 and parameters: {'n_hidden_layers': 4, 'n_neurons': 48, 'n_epochs': 10, 'optimizer': 'adam', 'learning_rate': 0.004379166866363832, 'dropout_rate': 0.5, 'batch_size': 128, 'weight_decay': 0.00010279920477210967}. Best is trial 0 with value: 0.796500027179718.
[I 2026-02-01 18:49:04,130] Trial 1 finished with value: 0.8495833277702332 and parameters: {'n_hidden_layers': 2, 'n_neurons': 56, 'n_epochs': 40, 'optimizer': 'rmsprop', 'learning_rate': 0.0067364554730414885, 'dropout_rate': 0.4, 'batch_size': 32, 'weight_decay': 5.952744169017467e-05}. Best is trial 1 with value: 0.8495833277702332.
[I 2026-02-01 18:50:27,744] Trial 2 finished with value: 0.8385833501815796 and parameters: {'n_hidden_layers': 1, 'n_neurons': 16, 'n_epochs': 30, 'optimizer': 'adam', 'learning_rate': 0.00015530218569260055, 'd

Best params : {'n_hidden_layers': 3, 'n_neurons': 120, 'n_epochs': 50, 'optimizer': 'sgd', 'learning_rate': 0.006527429939807163, 'dropout_rate': 0.1, 'batch_size': 32, 'weight_decay': 0.0004887828749303906}
Best Accuracy : 0.8955833315849304


In [17]:
# recreating network with best params

best_params = study.best_params

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.CrossEntropyLoss()

model = Network(
    input_dim=784,
    output_dim=10,
    n_hidden_layers=best_params['n_hidden_layers'],
    n_neurons=best_params['n_neurons'],
    dropout_rate=best_params['dropout_rate']
).to(device)

In [18]:
# optimizer with best params

if best_params['optimizer'] == 'adam':
    optimizer = optim.Adam(
        model.parameters(),
        lr=best_params['learning_rate'],
        weight_decay=best_params['weight_decay']
    )
elif best_params['optimizer'] == 'sgd':
    optimizer = optim.SGD(
        model.parameters(),
        lr=best_params['learning_rate'],
        weight_decay=best_params['weight_decay']
    )
else:
    optimizer = optim.RMSprop(
        model.parameters(),
        lr=best_params['learning_rate'],
        weight_decay=best_params['weight_decay']
    )

In [19]:
# retaining with best params

full_train_ds = CustomDataset(X, y)

train_loader = DataLoader(
    full_train_ds,
    batch_size=best_params['batch_size'],
    shuffle=True
)

for epoch in range(best_params['n_epochs']):
    model.train()
    total_loss = 0

    for batch_X, batch_y in train_loader:
        batch_X = batch_X.view(-1, 784).to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()
        logits = model(batch_X)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {total_loss/len(train_loader):.4f}")

Epoch 1 Loss: 0.7005
Epoch 2 Loss: 0.4765
Epoch 3 Loss: 0.4321
Epoch 4 Loss: 0.4059
Epoch 5 Loss: 0.3843
Epoch 6 Loss: 0.3665
Epoch 7 Loss: 0.3549
Epoch 8 Loss: 0.3458
Epoch 9 Loss: 0.3320
Epoch 10 Loss: 0.3281
Epoch 11 Loss: 0.3180
Epoch 12 Loss: 0.3122
Epoch 13 Loss: 0.3046
Epoch 14 Loss: 0.3010
Epoch 15 Loss: 0.2917
Epoch 16 Loss: 0.2878
Epoch 17 Loss: 0.2828
Epoch 18 Loss: 0.2771
Epoch 19 Loss: 0.2734
Epoch 20 Loss: 0.2705
Epoch 21 Loss: 0.2647
Epoch 22 Loss: 0.2620
Epoch 23 Loss: 0.2587
Epoch 24 Loss: 0.2534
Epoch 25 Loss: 0.2529
Epoch 26 Loss: 0.2519
Epoch 27 Loss: 0.2471
Epoch 28 Loss: 0.2425
Epoch 29 Loss: 0.2386
Epoch 30 Loss: 0.2366
Epoch 31 Loss: 0.2331
Epoch 32 Loss: 0.2322
Epoch 33 Loss: 0.2327
Epoch 34 Loss: 0.2250
Epoch 35 Loss: 0.2239
Epoch 36 Loss: 0.2229
Epoch 37 Loss: 0.2212
Epoch 38 Loss: 0.2233
Epoch 39 Loss: 0.2183
Epoch 40 Loss: 0.2153
Epoch 41 Loss: 0.2100
Epoch 42 Loss: 0.2086
Epoch 43 Loss: 0.2088
Epoch 44 Loss: 0.2067
Epoch 45 Loss: 0.2051
Epoch 46 Loss: 0.20

In [20]:
# loading test data

test_df = pd.read_csv("/content/drive/MyDrive/fmnist_full_dataset/fashion-mnist_test.csv")

In [21]:
X_test = test_df.drop('label',axis=1).values
y_test = test_df['label'].values

In [22]:
test_ds = CustomDataset(X_test, y_test)

test_loader = DataLoader(test_ds, batch_size=best_params['batch_size'])

In [23]:
from torchmetrics import Accuracy

accuracy = Accuracy(task="multiclass", num_classes=10).to(device)

model.eval()

accuracy.reset()

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X = batch_X.view(-1, 784).to(device)
        batch_y = batch_y.to(device)

        logits = model(batch_X)
        preds = torch.argmax(logits, dim=1)
        accuracy.update(preds, batch_y)

test_acc = accuracy.compute().item()

print(f"\nTest Accuracy: {test_acc:.4f}")


Test Accuracy: 0.7431


In [24]:
# saving model

torch.save(model.state_dict(), "best_fmnist_model.pth")

In [25]:
# loading model

# model.load_state_dict(torch.load("best_fmnist_model.pth"))
# model.eval()